# Word Embeddings — Extended Practical Notebook

A step-by-step NLP lesson from sparse text features to dense word vectors. This notebook covers one-hot encoding, vocabulary indexes, embedding matrices, cosine similarity, sentence pooling, context windows, optional Word2Vec training with Gensim, and static versus contextual embeddings.

**Note:** handwritten vectors are illustrative values, not pretrained embeddings. The tiny Word2Vec corpus only demonstrates the API; it cannot produce reliable general-purpose vectors.

## Learning objectives

- Explain why text needs numerical representations.
- Compare one-hot, Bag of Words, TF-IDF, and dense embeddings.
- Understand embedding dimensions and matrix lookup.
- Calculate cosine similarity and find nearest neighbours.
- Build sentence vectors by averaging token vectors.
- Understand context windows and Word2Vec's CBOW/Skip-gram intuition.
- Distinguish static embeddings from contextual transformer representations.
- Recognize limitations, bias, and evaluation considerations.

## 1. Why convert words to numbers?

Models need numerical inputs. A token ID is only a label: if `cat=0` and `dog=1`, the value 1 does not mean dog is more similar to banana or cat. An embedding instead represents a token with a vector whose values are learned from data.

In [1]:
words = ["cat", "dog", "banana"]
word_to_id = {word: index for index, word in enumerate(words)}
print(word_to_id)
print("Token IDs are lookup labels, not measures of meaning.")

{'cat': 0, 'dog': 1, 'banana': 2}
Token IDs are lookup labels, not measures of meaning.

## 2. One-hot encoding

One-hot encoding assigns each vocabulary word a vector containing one 1 and otherwise 0s. It represents identity, not semantic similarity. With a vocabulary of size V, each one-hot vector has V entries.

In [2]:
vocabulary = ["cat", "dog", "banana"]
one_hot_vectors = {
    word: [int(token == word) for token in vocabulary]
    for word in vocabulary
}
for word, vector in one_hot_vectors.items():
    print(f"{word:>7} -> {vector}")

    cat -> [1, 0, 0]
    dog -> [0, 1, 0]
 banana -> [0, 0, 1]

In [3]:
cat_dog_dot = sum(a * b for a, b in zip(one_hot_vectors["cat"], one_hot_vectors["dog"]))
cat_cat_dot = sum(a * b for a, b in zip(one_hot_vectors["cat"], one_hot_vectors["cat"]))
print("cat · dog =", cat_dog_dot)
print("cat · cat =", cat_cat_dot)

cat · dog = 0
cat · cat = 1

## 3. Bag of Words and TF-IDF recap

Bag of Words counts tokens in a document. TF-IDF weights tokens using their frequency in a document and their rarity across documents. These are useful sparse baselines, but they do not usually learn a dense vector for each word.

In [4]:
from collections import Counter
import re

documents = ["cats chase mice", "dogs chase cats", "ripe banana tastes sweet"]
tokens_by_document = [re.findall(r"[a-z]+", doc.lower()) for doc in documents]
vocabulary = sorted({token for tokens in tokens_by_document for token in tokens})
print("Vocabulary:", vocabulary)
for doc, tokens in zip(documents, tokens_by_document):
    counts = Counter(tokens)
    print(doc, "->", [counts[word] for word in vocabulary])

Vocabulary: ['banana', 'cats', 'chase', 'dogs', 'mice', 'ripe', 'sweet', 'tastes']
cats chase mice -> [0, 1, 1, 0, 1, 0, 0, 0]
dogs chase cats -> [0, 1, 1, 1, 0, 0, 0, 0]
ripe banana tastes sweet -> [1, 0, 0, 0, 0, 1, 1, 1]

## 4. Dense word embeddings

A dense embedding maps each word to a compact vector, perhaps 50, 100, or 300 dimensions. Unlike one-hot vectors, many entries may be non-zero. In learned embeddings, meaning is distributed across dimensions; individual dimensions usually do not have simple, fixed human-readable labels.

In [5]:
toy_embeddings = {
    "cat":    [0.80, 0.20, 0.10, 0.70],
    "kitten": [0.75, 0.25, 0.15, 0.72],
    "dog":    [0.70, 0.30, 0.20, 0.65],
    "banana": [0.10, 0.80, 0.90, 0.10],
}
for word, vector in toy_embeddings.items():
    print(f"{word:>7}: {vector} | dimension={len(vector)}")

    cat: [0.8, 0.2, 0.1, 0.7] | dimension=4
 kitten: [0.75, 0.25, 0.15, 0.72] | dimension=4
    dog: [0.7, 0.3, 0.2, 0.65] | dimension=4
 banana: [0.1, 0.8, 0.9, 0.1] | dimension=4

## 5. Embedding matrix and token lookup

For vocabulary size V and embedding dimension d, an embedding matrix has shape `(V, d)`. Each row stores the vector for one token ID. The integer ID is used to retrieve the row; it does not itself carry meaning.

In [6]:
vocabulary = ["banana", "cat", "dog", "kitten"]
word_to_index = {word: i for i, word in enumerate(vocabulary)}
index_to_word = {i: word for word, i in word_to_index.items()}
embedding_matrix = [
    [0.10, 0.80, 0.90],
    [0.80, 0.20, 0.10],
    [0.70, 0.30, 0.20],
    [0.75, 0.25, 0.15],
]
word = "dog"
word_id = word_to_index[word]
print("Word-to-index:", word_to_index)
print("Word:", word, "| ID:", word_id)
print("Vector lookup:", embedding_matrix[word_id])
print("Reverse lookup:", index_to_word[word_id])
print("Matrix shape:", (len(embedding_matrix), len(embedding_matrix[0])))

Word-to-index: {'banana': 0, 'cat': 1, 'dog': 2, 'kitten': 3}
Word: dog | ID: 2
Vector lookup: [0.7, 0.3, 0.2]
Reverse lookup: dog
Matrix shape: (4, 3)

## 6. Vector arithmetic

Vectors can be added, subtracted, and scaled. Such operations are mathematically straightforward, but a semantic interpretation only makes sense when vectors were trained appropriately. Do not infer linguistic rules from the handwritten values in this notebook.

In [7]:
vector_a = [1.0, 2.0, 3.0]
vector_b = [0.5, 1.0, 1.5]
added = [a + b for a, b in zip(vector_a, vector_b)]
subtracted = [a - b for a, b in zip(vector_a, vector_b)]
scaled = [2 * value for value in vector_a]
print("a + b =", added)
print("a - b =", subtracted)
print("2 * a =", scaled)

a + b = [1.5, 3.0, 4.5]
a - b = [0.5, 1.0, 1.5]
2 * a = [2.0, 4.0, 6.0]

## 7. Cosine similarity

For non-zero vectors a and b, cosine similarity is the dot product divided by the product of their lengths. Similar directions yield scores near 1, perpendicular directions near 0, and opposite directions near -1. Interpret scores relative to the embedding model and task.

In [8]:
from math import sqrt

a = [1, 2]
b = [2, 4]
dot_ab = sum(x * y for x, y in zip(a, b))
norm_a = sqrt(sum(x * x for x in a))
norm_b = sqrt(sum(y * y for y in b))
cos_ab = dot_ab / (norm_a * norm_b)
print("Dot product:", dot_ab)
print("Norm a:", round(norm_a, 4))
print("Norm b:", round(norm_b, 4))
print("Cosine similarity:", round(cos_ab, 4))

Dot product: 10
Norm a: 2.2361
Norm b: 4.4721
Cosine similarity: 1.0

In [9]:
for first, second in [("cat", "kitten"), ("cat", "dog"), ("cat", "banana")]:
    vector_a = toy_embeddings[first]
    vector_b = toy_embeddings[second]
    dot = sum(x * y for x, y in zip(vector_a, vector_b))
    norm_a = sqrt(sum(x * x for x in vector_a))
    norm_b = sqrt(sum(y * y for y in vector_b))
    score = dot / (norm_a * norm_b)
    print(f"{first:>6} vs {second:<7}: {score:.4f}")

   cat vs kitten : 0.9979
   cat vs dog    : 0.9956
   cat vs banana : 0.3038

## 8. Find nearest neighbours

Nearest-neighbour search ranks candidate vectors by similarity. Large-scale systems often use specialized vector search indexes; this small function is enough for a basic example.

In [10]:
query_word = "cat"
query_vector = toy_embeddings[query_word]
similarity_scores = []
for candidate, candidate_vector in toy_embeddings.items():
    if candidate != query_word:
        dot = sum(x * y for x, y in zip(query_vector, candidate_vector))
        norm_q = sqrt(sum(x * x for x in query_vector))
        norm_c = sqrt(sum(x * x for x in candidate_vector))
        similarity_scores.append((candidate, dot / (norm_q * norm_c)))
similarity_scores = sorted(similarity_scores, key=lambda item: item[1], reverse=True)
for word, score in similarity_scores:
    print(f"{word:>7}: {score:.4f}")

 kitten: 0.9979
    dog: 0.9956
 banana: 0.3038

## 9. Sentence vectors using mean pooling

A simple sentence representation is the average of its word vectors. It produces a fixed-size vector for sentences of different lengths. However, averaging loses word order and may blur negation or different word senses.

In [11]:
sentence_words = ["cat", "kitten"]
known_vectors = [toy_embeddings[word] for word in sentence_words if word in toy_embeddings]
sentence_vector = [
    sum(vector[i] for vector in known_vectors) / len(known_vectors)
    for i in range(len(known_vectors[0]))
]
print("Words:", sentence_words)
print("Mean sentence vector:", sentence_vector)

Words: ['cat', 'kitten']
Mean sentence vector: [0.775, 0.225, 0.125, 0.71]

### Weighted averaging

A weighted average gives some tokens more influence than others. TF-IDF weights are one possible baseline. Weighted averaging is still not the same as a dedicated, trained sentence embedding model.

In [12]:
sentence_words = ["cat", "kitten", "dog"]
token_weights = {"cat": 2.0, "kitten": 1.5, "dog": 1.0}
weighted_vectors = [(toy_embeddings[word], token_weights.get(word, 1.0))
                    for word in sentence_words if word in toy_embeddings]
total_weight = sum(weight for _, weight in weighted_vectors)
weighted_sentence_vector = [
    sum(vector[i] * weight for vector, weight in weighted_vectors) / total_weight
    for i in range(len(weighted_vectors[0][0]))
]
print("Weighted sentence vector:", [round(value, 4) for value in weighted_sentence_vector])

Weighted sentence vector: [0.7278, 0.2722, 0.1611, 0.6756]

## 10. Context windows

Embedding methods use context patterns. A context window defines how many nearby tokens are considered around a target word. The example below generates target-context pairs within a symmetric window.

In [13]:
tokens = ["cats", "chase", "small", "mice"]
window_size = 2
context_pairs = []
for i, target in enumerate(tokens):
    start = max(0, i - window_size)
    end = min(len(tokens), i + window_size + 1)
    for j in range(start, end):
        if i != j:
            context_pairs.append((target, tokens[j]))
for pair in context_pairs:
    print(pair)

('cats', 'chase')
('cats', 'small')
('chase', 'cats')
('chase', 'small')
('chase', 'mice')
('small', 'cats')
('small', 'chase')
('small', 'mice')
('mice', 'chase')
('mice', 'small')

## 11. Optional: train Word2Vec with Gensim

Install Gensim in the same Python environment as your notebook kernel:

```bash
python -m pip install gensim
```

If the import fails, check the active interpreter with `import sys; print(sys.executable)`. The tiny sample corpus below only demonstrates the API. Its learned vectors are not reliable general-purpose embeddings.

In [14]:
raw_sentences = [
    "cats chase small mice",
    "dogs chase small cats",
    "mice hide in small spaces",
    "dogs and cats are animals",
    "kittens are young cats",
    "puppies are young dogs",
    "cats and dogs can be pets",
    "small animals need food",
]
tokenized_sentences = [re.findall(r"[a-z]+", sentence.lower()) for sentence in raw_sentences]
for sentence in tokenized_sentences:
    print(sentence)

['cats', 'chase', 'small', 'mice']
['dogs', 'chase', 'small', 'cats']
['mice', 'hide', 'in', 'small', 'spaces']
['dogs', 'and', 'cats', 'are', 'animals']
['kittens', 'are', 'young', 'cats']
['puppies', 'are', 'young', 'dogs']
['cats', 'and', 'dogs', 'can', 'be', 'pets']
['small', 'animals', 'need', 'food']

In [15]:
try:
    from gensim.models import Word2Vec
    gensim_available = True
    print("Gensim is available.")
except ImportError:
    gensim_available = False
    print("Gensim is not installed in this kernel.")
    print("Install with: python -m pip install gensim")

Gensim is not installed in this kernel.
Install with: python -m pip install gensim

In [16]:
if gensim_available:
    w2v_model = Word2Vec(
        sentences=tokenized_sentences,
        vector_size=30,
        window=2,
        min_count=1,
        workers=1,
        sg=1,
        seed=42,
        epochs=100,
    )
    print("Vocabulary size:", len(w2v_model.wv))
    print("Vector shape:", w2v_model.wv["cats"].shape)
else:
    print("Skipped: install Gensim to train Word2Vec.")

Skipped: install Gensim to train Word2Vec.

In [17]:
if gensim_available:
    print("Nearest words to cats:")
    for word, score in w2v_model.wv.most_similar("cats", topn=5):
        print(f"{word:>10}: {score:.4f}")
else:
    print("Skipped: no trained Word2Vec model.")

Skipped: no trained Word2Vec model.

In [18]:
if gensim_available:
    for first, second in [("cats", "dogs"), ("cats", "kittens"), ("cats", "banana")]:
        if first in w2v_model.wv and second in w2v_model.wv:
            print(first, "vs", second, round(w2v_model.wv.similarity(first, second), 4))
else:
    print("Skipped: no trained Word2Vec model.")

Skipped: no trained Word2Vec model.

### CBOW vs Skip-gram

- **CBOW:** predicts a target word from surrounding context words.
- **Skip-gram:** predicts surrounding context words from a target word.

In Gensim's `Word2Vec`, `sg=0` selects CBOW and `sg=1` selects Skip-gram. The separate [Word2Vec lesson](../05_Word2Vec/README.md) explores these models in more depth.

In [19]:
if gensim_available:
    cbow_model = Word2Vec(
        sentences=tokenized_sentences,
        vector_size=30,
        window=2,
        min_count=1,
        workers=1,
        sg=0,
        seed=42,
        epochs=100,
    )
    print("CBOW vector shape:", cbow_model.wv["cats"].shape)
else:
    print("Skipped: install Gensim to run CBOW.")

Skipped: install Gensim to run CBOW.

## 12. Static vs contextual embeddings

Word2Vec and GloVe usually provide a single vector per vocabulary word. This is called a static embedding. The word `bank` therefore has one vector in both financial and river contexts.

Transformer models use surrounding tokens to build contextual representations. The representation of `bank` can differ between “I deposited money at the bank” and “We sat on the river bank.” The exact embedding depends on the model, tokenizer, layer, and pooling method.

In [20]:
examples = [
    "I deposited money at the bank.",
    "We sat on the river bank.",
]
for sentence in examples:
    print(sentence)
print("\nStatic embedding: one stored vector for 'bank'.")
print("Contextual embedding: representation can change with surrounding words.")

I deposited money at the bank.
We sat on the river bank.

Static embedding: one stored vector for 'bank'.
Contextual embedding: representation can change with surrounding words.

## 13. Compare representation methods

| Method | Representation | Advantage | Limitation |
|---|---|---|---|
| One-hot | Sparse token vector | Simple identity representation | No semantic closeness |
| Bag of Words | Sparse document counts | Interpretable baseline | Usually ignores word order |
| TF-IDF | Sparse weighted terms | Highlights informative words | Limited semantic generalization |
| Word2Vec | Static dense word vector | Learns usage patterns | One vector per word |
| GloVe | Static dense word vector | Uses global co-occurrence | One vector per word |
| Transformer | Contextual token vectors | Context-sensitive representation | More computationally expensive |
| Sentence embedding model | Whole-sentence vector | Useful for semantic search | Quality depends on model/task |

## 14. Applications

Embeddings are used for semantic search, retrieval, text classification, clustering, duplicate-question detection, recommendation, document matching, and as inputs to neural NLP systems. In retrieval-augmented generation, an embedding model can help find relevant documents before a language model generates a response.

An embedding alone is not a complete system: the dataset, downstream model, evaluation metric, and deployment context all matter.

## 15. Limitations and evaluation

- **Data dependence:** vectors reflect patterns in the training corpus.
- **Bias:** social and cultural biases can be encoded in representations.
- **Vocabulary coverage:** older static models may not represent unseen words well.
- **Polysemy:** one static vector can mix multiple meanings.
- **Domain shift:** vectors learned on one domain may perform poorly on another.
- **Similarity is not truth:** related words, synonyms, and opposites can all occur in similar contexts.
- **Tiny corpus warning:** toy data is not sufficient for production quality.

For a real project, define the task, compare against a TF-IDF baseline, evaluate on held-out data, inspect nearest neighbours, check bias, and record preprocessing and model versions.

## 16. Summary and references

Remember: embeddings are dense vectors learned from data; token IDs are lookup indices; cosine similarity compares vector directions; averaging word vectors loses order; Word2Vec/GloVe are usually static; transformer representations are contextual.

References:
- [Word2Vec paper](https://arxiv.org/abs/1301.3781)
- [GloVe project](https://nlp.stanford.edu/projects/glove/)
- [Gensim Word2Vec documentation](https://radimrehurek.com/gensim/models/word2vec.html)
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- [scikit-learn cosine similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)